In [2]:
import pandas as pd

df = pd.read_csv("data/processed/loans_with_target.csv")

print("Shape:", df.shape)
df.head()

Shape: (1348099, 49)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,mo_sin_old_rev_tl_op,mort_acc,num_actv_bc_tl,num_bc_sats,num_sats,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit,default
0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,...,128.0,1.0,2.0,2.0,7.0,0.0,0.0,7746.0,2400.0,0
1,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,...,192.0,4.0,5.0,13.0,22.0,0.0,0.0,39475.0,79300.0,0
2,20000.0,60 months,10.78,432.66,B,B4,truck driver,10+ years,MORTGAGE,63000.0,...,184.0,5.0,2.0,2.0,6.0,0.0,0.0,18696.0,6200.0,0
3,10400.0,60 months,22.45,289.91,F,F1,Contract Specialist,3 years,MORTGAGE,104433.0,...,210.0,6.0,4.0,5.0,12.0,0.0,0.0,95768.0,20300.0,0
4,11950.0,36 months,13.44,405.18,C,C3,Veterinary Tecnician,4 years,RENT,34000.0,...,54.0,0.0,2.0,2.0,5.0,0.0,0.0,12798.0,9400.0,0


In [3]:
df['term'].unique()

<ArrowStringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str

In [4]:
# Extract the numeric part of term and convert to integer
df['term'] = df['term'].str.strip()          # remove leading/trailing spaces
df['term'] = df['term'].str.replace(' months', '', regex=False)  # remove the word "months"
df['term'] = df['term'].astype(int)           # convert from text to actual integer
print(df['term'].unique())
print(df['term'].dtype)

[36 60]
int64


In [5]:
df['emp_length'].unique()

<ArrowStringArray>
['10+ years',   '3 years',   '4 years',   '6 years',   '7 years',   '8 years',
   '2 years',   '5 years',   '9 years',  '< 1 year',    '1 year',         nan]
Length: 12, dtype: str

In [6]:
# Step 1: Define a function that converts each raw text value to a number
def clean_emp_length(value):
    if pd.isna(value):
        return -1  # our "missing" indicator
    elif value == '< 1 year':
        return 0
    elif value == '10+ years':
        return 10
    else:
        # For values like '3 years', '7 years' — extract just the number
        return int(value.split()[0])

# Step 2: Apply this function to every row in the column
df['emp_length'] = df['emp_length'].apply(clean_emp_length)

# Step 3: Verify it worked
print(df['emp_length'].unique())
print(df['emp_length'].value_counts().sort_index())

[10  3  4  6  7  8  2  5  9  0  1 -1]
emp_length
-1      78550
 0     108537
 1      88843
 2     122100
 3     107868
 4      80763
 5      84326
 6      62879
 7      59724
 8      60811
 9      51019
 10    442679
Name: count, dtype: int64


In [7]:
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_percent = (missing_summary / len(df) * 100).round(2)

print(missing_percent)

emp_title                     6.38
avg_cur_bal                   5.21
mo_sin_old_rev_tl_op          5.21
tot_coll_amt                  5.21
num_actv_bc_tl                5.21
total_rev_hi_lim              5.21
tot_cur_bal                   5.21
bc_util                       4.80
bc_open_to_buy                4.74
num_bc_sats                   4.35
num_sats                      4.35
total_bc_limit                3.71
mort_acc                      3.71
acc_open_past_24mths          3.71
total_bal_ex_mort             3.71
title                         1.24
pub_rec_bankruptcies          0.10
revol_util                    0.07
dti                           0.03
collections_12_mths_ex_med    0.01
chargeoff_within_12_mths      0.01
tax_liens                     0.01
inq_last_6mths                0.00
total_acc                     0.00
delinq_2yrs                   0.00
earliest_cr_line              0.00
acc_now_delinq                0.00
open_acc                      0.00
pub_rec             

In [8]:
# --- Group A: Drop free-text columns we've decided not to use ---
df = df.drop(columns=['emp_title', 'title'])

# --- Group B: Fill numeric credit-history columns with median ---
group_b_columns = [
    'avg_cur_bal', 'mo_sin_old_rev_tl_op', 'tot_coll_amt', 'num_actv_bc_tl',
    'total_rev_hi_lim', 'tot_cur_bal', 'bc_util', 'bc_open_to_buy',
    'num_bc_sats', 'num_sats', 'total_bc_limit', 'mort_acc',
    'acc_open_past_24mths', 'total_bal_ex_mort'
]

for col in group_b_columns:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# --- Group C: Drop rows with tiny amounts of missing data ---
group_c_columns = [
    'pub_rec_bankruptcies', 'revol_util', 'dti', 'collections_12_mths_ex_med',
    'chargeoff_within_12_mths', 'tax_liens'
]

df = df.dropna(subset=group_c_columns)

# --- Verify: check if any missing values remain ---
print("Shape after cleaning:", df.shape)
print("\nRemaining missing values:")
print(df.isnull().sum().sum())  # should be 0

Shape after cleaning: (1345491, 47)

Remaining missing values:
2


In [9]:
# Find which column(s) still have missing values, and how many
remaining_missing = df.isnull().sum()
print(remaining_missing[remaining_missing > 0])

zip_code          1
inq_last_6mths    1
dtype: int64


In [10]:
df = df.dropna()

# Final verification
print("Final shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

Final shape: (1345489, 47)
Remaining missing values: 0


In [11]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

print(df[['issue_d', 'earliest_cr_line']].head())
print(df['issue_d'].dtype)

     issue_d earliest_cr_line
0 2015-12-01       2003-08-01
1 2015-12-01       1999-12-01
2 2015-12-01       2000-08-01
3 2015-12-01       1998-06-01
4 2015-12-01       1987-10-01
datetime64[us]


In [12]:
# Calculate credit history length in years
df['credit_history_years'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25

print(df['credit_history_years'].describe())

count    1.345489e+06
mean     1.625644e+01
std      7.507034e+00
min      1.002053e+00
25%      1.124435e+01
50%      1.474880e+01
75%      2.000000e+01
max      8.324983e+01
Name: credit_history_years, dtype: float64


In [13]:
df = df.drop(columns=['issue_d', 'earliest_cr_line'])

print("Shape after dropping raw date columns:", df.shape)

Shape after dropping raw date columns: (1345489, 46)


In [14]:
categorical_cols = ['grade', 'sub_grade', 'home_ownership', 'verification_status', 
                     'purpose', 'addr_state', 'zip_code']

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].unique()[:10])  # show first 10 unique values as a sample
    print()

grade: 7 unique values
<ArrowStringArray>
['C', 'B', 'F', 'A', 'E', 'D', 'G']
Length: 7, dtype: str

sub_grade: 35 unique values
<ArrowStringArray>
['C4', 'C1', 'B4', 'F1', 'C3', 'B2', 'B1', 'A2', 'B5', 'C2']
Length: 10, dtype: str

home_ownership: 6 unique values
<ArrowStringArray>
['MORTGAGE', 'RENT', 'OWN', 'ANY', 'NONE', 'OTHER']
Length: 6, dtype: str

verification_status: 3 unique values
<ArrowStringArray>
['Not Verified', 'Source Verified', 'Verified']
Length: 3, dtype: str

purpose: 14 unique values
<ArrowStringArray>
['debt_consolidation',     'small_business',   'home_improvement',
     'major_purchase',        'credit_card',              'other',
              'house',           'vacation',                'car',
            'medical']
Length: 10, dtype: str

addr_state: 51 unique values
<ArrowStringArray>
['PA', 'SD', 'IL', 'GA', 'MN', 'SC', 'RI', 'NC', 'CA', 'VA']
Length: 10, dtype: str

zip_code: 945 unique values
<ArrowStringArray>
['190xx', '577xx', '605xx', '174xx', '300

In [15]:
# Grade: A (best) to G (worst) -> 0 to 6
grade_order = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['grade'] = df['grade'].map(grade_order)

# Sub_grade: A1 (best) to G5 (worst) -> 0 to 34
sub_grades_sorted = sorted(df['sub_grade'].unique())  # e.g. ['A1','A2',...,'G5']
sub_grade_order = {sg: i for i, sg in enumerate(sub_grades_sorted)}
df['sub_grade'] = df['sub_grade'].map(sub_grade_order)

print(df[['grade', 'sub_grade']].head())

   grade  sub_grade
0      2         13
1      2         10
2      1          8
3      5         25
4      2         12


In [16]:
df = pd.get_dummies(df, columns=['home_ownership', 'verification_status', 'purpose'], 
                     drop_first=True)

print("Shape after one-hot encoding:", df.shape)

Shape after one-hot encoding: (1345489, 63)


In [17]:
# Frequency encoding: replace each state with how often it appears
state_freq = df['addr_state'].value_counts(normalize=True)
df['addr_state'] = df['addr_state'].map(state_freq)

# Drop zip_code entirely
df = df.drop(columns=['zip_code'])

print("Final shape:", df.shape)
df.head()

Final shape: (1345489, 62)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,annual_inc,loan_status,addr_state,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,3600.0,36,13.99,123.03,2,13,10,55000.0,Fully Paid,0.033883,...,False,False,False,False,False,False,False,False,False,False
1,24700.0,36,11.99,820.28,2,10,10,65000.0,Fully Paid,0.002054,...,False,False,False,False,False,False,False,True,False,False
2,20000.0,60,10.78,432.66,1,8,10,63000.0,Fully Paid,0.038509,...,True,False,False,False,False,False,False,False,False,False
3,10400.0,60,22.45,289.91,5,25,3,104433.0,Fully Paid,0.033883,...,False,False,True,False,False,False,False,False,False,False
4,11950.0,36,13.44,405.18,2,12,4,34000.0,Fully Paid,0.032227,...,False,False,False,False,False,False,False,False,False,False


In [18]:
df = df.drop(columns=['loan_status'])
print("Final shape:", df.shape)

Final shape: (1345489, 61)


In [19]:
X = df.drop(columns=['default'])
y = df['default']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1345489, 60)
y shape: (1345489,)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining set default rate:", y_train.mean())
print("Test set default rate:", y_test.mean())

Training set shape: (1076391, 60)
Test set shape: (269098, 60)

Training set default rate: 0.19975826628056162
Test set default rate: 0.19975993875837056


In [22]:
X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

print("All splits saved!")


All splits saved!
